<a href="https://colab.research.google.com/github/Thrivi17/Thrivi_Flyrank/blob/main/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Task Framing

# Chosen Lane

* Lane Choice: CTR / Engagement Opportunity Scoring

# Task Type

* Task Type: Scoring & Ranking

* Explanation:

* The primary functionality of this system is scoring - specifically, it's trained to predict an expected baseline CTR (Click Through Rate) or an explicit “opportunity score” on a given page-query combination.
 * This score is then treated as an opportunity, and the resulting pair is ranked from highest to lowest expected traffic lift. The teams (Engineering and Content) then use this rank to prioritize which pages and queries they’ll spend their time and effort on improving.

# Target or Proxy Variable

* Target/Proxy: CTR Deviation (Expected CTR minus Actual CTR) weighted by Impressions.

* Explanation:

* The model is intended to learn an estimate for what the expected CTR for that specific page and query should be given its current rank in the search engine and other context like the user device.
 * The machine learning “opportunity” metric is simply the difference between the learned Expected CTR and the Actual CTR that the pair achieved in organic results, then multiplied by the number of impressions that the page and query generated so it capture the “amount of lost opportunity”.

# Success Metric

* Offline ML Metric: Mean Absolute Error (MAE) or Root Mean Squared Error (RMSE) between the Predicted Expected CTR and the actual observed Baseline CTR for well-performing typical page-queries.
* Business Metric: Traffic Lift@50 (The percentage change in organic traffic the team can achieve from the top 50 underperforming page-query opportunities, using titles, metas or layouts)

# Action the Output Supports

* Actionable Impact:

* This system outputs a list of the Top 50 Page/Queries each week with the highest opportunity. The team can quickly act on those with title, meta, or page-layout changes to reclaim traffic that the machine learning system predicts they should be capturing.

# Why Machine Learning Beats a Fixed Rule

* Non-linear Dynamics:

* The relationships within this dataset are far too non-linear to be effectively captured by a simple, fixed set of rules. The average CTR drops off dramatically (and unevenly!) as a search position progresses – from an average 6.6694% CTR at position 1.0, it goes down to 2.3798% at position 2.0 and then down to a paltry 0.3056% by position 10.0.

* Failure of Linear Rules:

* In contrast to these dynamics, the overall linear correlation between position and CTR is practically non-existent (only -0.0726). Simply using this single feature with linear regression or hard-coding a formula would fail completely to account for the complexities of the interactions and yield wildly inaccurate predictions.

In [4]:
import numpy as np
import pandas as pd

# 1. Load Lane's Slice of Starter Data
data_slice = {
    "page_url": [
        "/blog/ml-basics",
        "/blog/ml-basics",
        "/store/shoes",
        "/services/consulting",
        "/blog/guide-to-sql",
    ],
    "search_query": [
        "what is machine learning",
        "ml course free",
        "buy running shoes",
        "enterprise tech help",
        "nested query tutorial",
    ],
    "impressions": [12000, 450, 8500, 3100, 950],
    "clicks": [110, 30, 560, 22, 3],
    "avg_position": [1.2, 2.1, 1.0, 3.4, 10.2],
}

df_raw = pd.DataFrame(data_slice)

# 2. Show the Unit of Analysis
df_raw["actual_ctr"] = (df_raw["clicks"] / df_raw["impressions"]) * 100

print("--- Unit of Analysis Verification ---")
print("Question: One row = one what?")
print("Answer:   One row = one unique page-query pair.\n")
print(f"DataFrame Shape: {df_raw.shape} (Rows, Columns)\n")
display(df_raw.head())

# 3. Sketch the Target/Opportunity Column
def sketch_expected_ctr(pos):
    if pos <= 1.5:
        return 6.6694
    elif pos <= 2.5:
        return 2.3798
    else:
        return 0.3056 + (10.0 / (pos + 1))

df_raw["expected_ctr_baseline"] = df_raw["avg_position"].apply(sketch_expected_ctr)

df_raw["ctr_gap"] = df_raw["expected_ctr_baseline"] - df_raw["actual_ctr"]
df_raw["ctr_gap"] = df_raw["ctr_gap"].clip(lower=0)

df_raw["traffic_opportunity_score"] = (df_raw["ctr_gap"] / 100) * df_raw["impressions"]

df_ranked = df_raw.sort_values(by="traffic_opportunity_score", ascending=False).reset_index(drop=True)

print("\n--- Final Target Column Sketch (Sorted by Priority) ---")
cols_to_show = ["page_url", "search_query", "avg_position", "actual_ctr", "expected_ctr_baseline", "traffic_opportunity_score"]
display(df_ranked[cols_to_show])

--- Unit of Analysis Verification ---
Question: One row = one what?
Answer:   One row = one unique page-query pair.

DataFrame Shape: (5, 6) (Rows, Columns)



,page_url,search_query,impressions,clicks,avg_position,actual_ctr
0,/blog/ml-basics,what is machine learning,12000,110,1.2,0.916667
1,/blog/ml-basics,ml course free,450,30,2.1,6.666667
2,/store/shoes,buy running shoes,8500,560,1.0,6.588235
3,/services/consulting,enterprise tech help,3100,22,3.4,0.709677
4,/blog/guide-to-sql,nested query tutorial,950,3,10.2,0.315789



--- Final Target Column Sketch (Sorted by Priority) ---


,page_url,search_query,avg_position,actual_ctr,expected_ctr_baseline,traffic_opportunity_score
0,/blog/ml-basics,what is machine learning,1.2,0.916667,6.669400,690.328000
1,/services/consulting,enterprise tech help,3.4,0.709677,2.578327,57.928145
2,/blog/guide-to-sql,nested query tutorial,10.2,0.315789,1.198457,8.385343
3,/store/shoes,buy running shoes,1.0,6.588235,6.669400,6.899000
4,/blog/ml-basics,ml course free,2.1,6.666667,2.379800,0.000000
